In [17]:
from pyspark.sql import SparkSession, Row
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType, LongType, StringType
from pyspark.sql.functions import col, greatest, count, desc, least, lit, isnan, when, count, explode, round as spark_round
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.ticker as mticker
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.feature import HashingTF, IDF, Tokenizer, StringIndexer
from pyspark.sql.functions import concat_ws, collect_list, lower, regexp_replace
from pyspark.ml import Pipeline
import numpy as np
from pyspark.ml.feature import Normalizer
from pyspark.ml.linalg import Vectors, SparseVector
from collections import defaultdict

## Setup

In [2]:
SMALL = "../data/processed/small/"
LARGE = "../data/processed/32m/"
DATA = SMALL

## Modélisation — ALS (Alternating Least Squares)

### Principe

ALS est un algorithme de **filtrage collaboratif par factorisation matricielle**.
Il décompose la matrice sparse users × films en deux matrices denses de dimension
réduite (`rank`), dont le produit reconstruit les notes manquantes.

Les **facteurs latents** capturent automatiquement des préférences implicites
(goût pour les films sombres, préférence pour certaines époques...) sans qu'on
les définisse manuellement.

### Hyperparamètres clés

| Paramètre | Rôle | Valeur choisie |
|---|---|---|
| `rank` | Nombre de facteurs latents | 10 |
| `maxIter` | Nombre d'itérations | 10 |
| `regParam` | Régularisation (évite l'overfitting) | 0.1 |
| `coldStartStrategy` | Gestion des users/films inconnus | `drop` |

`coldStartStrategy="drop"` : si un user ou film du jeu de test n'apparaît pas
dans le train, ALS ne peut pas prédire — on supprime ces lignes plutôt que
d'obtenir `NaN` dans l'évaluation.

In [3]:
spark = SparkSession.builder \
    .appName("SparkleMovie-Modelling") \
    .master("local[*]") \
    .config("spark.driver.memory", "10g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark version : {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/16 17:51:17 WARN Utils: Your hostname, MacBook-M4-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.10.42.119 instead (on interface en0)
26/03/16 17:51:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/16 17:51:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/16 17:51:18 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version : 4.1.1


In [4]:
# Consistent style across all plots
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "white",
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "axes.grid":        True,
    "grid.alpha":       0.3,
    "font.size":        11,
})

In [5]:
ratings_clean      = spark.read.parquet(f"{DATA}ratings_clean.parquet")
movies_clean       = spark.read.parquet(f"{DATA}movies_clean.parquet")
movies_with_genres = spark.read.parquet(f"{DATA}movies_with_genres.parquet")

print(f"ratings_clean      : {ratings_clean.count():,} rows")
print(f"movies_clean       : {movies_clean.count():,} rows")
print(f"movies_with_genres : {movies_with_genres.count():,} rows")

ratings_clean      : 100,836 rows
movies_clean       : 9,742 rows
movies_with_genres : 9,708 rows


In [6]:
train, test = ratings_clean.randomSplit([0.8, 0.2], seed=42)

print(f"Train : {train.count():,} rows")
print(f"Test  : {test.count():,} rows")

# Build ALS model
als = ALS(
    rank=10,
    maxIter=10,
    regParam=0.1,
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    coldStartStrategy="drop",
    seed=42
    # nonnegative=True  # Non negative factors (optional, can improve interpretability)
)

# Train
model = als.fit(train)
print("Model trained ✓")

Train : 80,578 rows
Test  : 20,258 rows
Model trained ✓


In [7]:
# Predict on test set
predictions = model.transform(test)

# RMSE
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

rmse = evaluator.evaluate(predictions)
print(f"RMSE (rank=10, maxIter=10, regParam=0.1) : {rmse:.4f}")

# Sample predictions vs actual
predictions.select("userId", "movieId", "rating", "prediction") \
            .orderBy("userId") \
            .show(10)

RMSE (rank=10, maxIter=10, regParam=0.1) : 0.8814
+------+-------+------+----------+
|userId|movieId|rating|prediction|
+------+-------+------+----------+
|     1|      6|   4.0| 4.5788817|
|     1|    943|   4.0| 3.8743846|
|     1|    101|   5.0|  4.217507|
|     1|    151|   5.0| 4.1573143|
|     1|    231|   5.0| 4.0079527|
|     1|    349|   4.0|  4.069611|
|     1|    423|   3.0| 2.7713876|
|     1|    543|   4.0| 4.2908926|
|     1|    596|   5.0| 4.2220583|
|     1|    923|   5.0|  4.381694|
+------+-------+------+----------+
only showing top 10 rows


In [8]:
# Parameter grid to explore
param_grid = (
    ParamGridBuilder()
    .addGrid(als.rank,     [5, 10, 15])
    .addGrid(als.regParam, [0.01, 0.1, 0.5])
    .build()
)

cv = CrossValidator(
    estimator=als,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=3,
    seed=42
)

cv_model = cv.fit(train)

# Best model params
best_model = cv_model.bestModel
print(f"Best rank     : {best_model.rank}")
print(f"Best regParam : {best_model._java_obj.parent().getRegParam()}")

# Predictions avec clipping 0.5–5.0 
best_predictions = (
    best_model.transform(test)
    .withColumn("prediction", greatest(lit(0.5), least(lit(5.0), col("prediction"))))
)

best_rmse = evaluator.evaluate(best_predictions)
print(f"Best RMSE (no leakage, clipped) : {best_rmse:.4f}")

# Vérification hors plage
out_of_range = best_predictions.filter(
    (col("prediction") < 0.5) | (col("prediction") > 5.0)
).count()
print(f"Predictions hors plage : {out_of_range}")

Best rank     : 5
Best regParam : 0.1
Best RMSE (no leakage, clipped) : 0.8748
Predictions hors plage : 0


## Résultats ALS

### Modèle initial

| Paramètre | Valeur |
|---|---|
| `rank` | 10 |
| `maxIter` | 10 |
| `regParam` | 0.1 |
| **RMSE** | **0.8814** |

Un RMSE de 0.88 signifie qu'en moyenne, la prédiction s'écarte de
0.88 étoile par rapport à la note réelle (sur une échelle de 0.5 à 5.0).

### Tuning par grid search (CrossValidator, 3 folds)

Grille explorée : `rank` ∈ {5, 10, 15} × `regParam` ∈ {0.01, 0.1, 0.5}
→ 9 combinaisons × 3 folds = 27 modèles entraînés.
Le CrossValidator est entraîné uniquement sur `train` — le test set
reste invisible pendant toute la phase de tuning.

| Paramètre | Valeur optimale |
|---|---|
| `rank` | 5 |
| `regParam` | 0.1 |
| **RMSE** | **0.8748** |

Les prédictions sont clippées entre 0.5 et 5.0 — ALS étant un modèle
de factorisation matricielle sans contrainte de bornes, il peut produire
des valeurs hors plage sans ce post-traitement.

### Interprétation

Le tuning améliore marginalement le RMSE (0.8814 → 0.8748, soit -0.007).
Un `rank=5` optimal suggère que le dataset small ne nécessite pas une
représentation complexe — 5 facteurs latents suffisent à capturer les
préférences des utilisateurs. Un `rank` trop élevé sur peu de données
risque l'overfitting, que `regParam=0.1` contribue à limiter.

In [9]:
# Pick 5 user IDs that exist in the dataset
sample_user_ids = [1, 42, 100, 200, 500]
sample_users_df = spark.createDataFrame(
    [Row(userId=uid) for uid in sample_user_ids]
)

# Get recommendations for these specific users
recs = best_model.recommendForUserSubset(sample_users_df, 10)

# Explode, clip predictions, and join with movie titles
recs_exploded = (
    recs
    .select("userId", explode("recommendations").alias("rec"))
    .select("userId", col("rec.movieId"), col("rec.rating").alias("predicted_rating"))
    .withColumn("predicted_rating", greatest(lit(0.5), least(lit(5.0), col("predicted_rating"))))
    .join(movies_clean.select("movieId", "title"), on="movieId", how="left")
    .orderBy("userId", desc("predicted_rating"))
)

recs_exploded.show(50, truncate=False)

+-------+------+----------------+-----------------------------------------------------------------------------------------------------------------------------+
|movieId|userId|predicted_rating|title                                                                                                                        |
+-------+------+----------------+-----------------------------------------------------------------------------------------------------------------------------+
|3266   |1     |5.0             |Man Bites Dog (C'est arrivé près de chez vous) (1992)                                                                        |
|8477   |1     |5.0             |Jetée, La (1962)                                                                                                             |
|25771  |1     |5.0             |Andalusian Dog, An (Chien andalou, Un) (1929)                                                                                |
|96004  |1     |5.0             |Dragon 

## Recommandation basée sur le contenu (Content-Based)

### Principe

Contrairement à ALS qui exploite le comportement collectif des utilisateurs,
l'approche content-based analyse la **description des films** pour recommander
des contenus similaires à ceux qu'un utilisateur a appréciés.

### Pipeline

1. Construction d'un profil textuel par film (genres + tags utilisateurs)
2. Vectorisation TF-IDF : chaque film devient un vecteur numérique
3. Calcul de similarité cosinus entre films
4. Pour chaque user : identifier ses films préférés (note ≥ 4.0),
   puis recommander les films les plus similaires non encore vus

### Pourquoi TF-IDF plutôt qu'un simple comptage ?

Un genre comme "Drama" apparaît dans 40% du catalogue — il est peu
discriminant. Un tag rare comme "existentialism" ou "twist ending"
est bien plus informatif. TF-IDF pondère automatiquement les termes
rares plus fortement que les termes fréquents.

In [10]:
# Load tags
tags = spark.read.parquet(f"{DATA}tags_clean.parquet")

# Aggregate tags per movie : one row per movie with all tags concatenated
tags_per_movie = (
    tags
    .groupBy("movieId")
    .agg(concat_ws(" ", collect_list(lower(col("tag")))).alias("tags_text"))
)

# Join movies with their tags
movies_with_content = (
    movies_with_genres
    .join(tags_per_movie, on="movieId", how="left")
    .fillna("", subset=["tags_text"])
)

# Build content field : genres (pipe → space) + tags
movies_with_content = movies_with_content.withColumn(
    "content",
    concat_ws(" ",
        regexp_replace(col("genres"), "\\|", " "),
        col("tags_text")
    )
)

movies_with_content.select("movieId", "title", "content").show(5, truncate=False)

+-------+----------------------------------+-----------------------------------------------------------------------+
|movieId|title                             |content                                                                |
+-------+----------------------------------+-----------------------------------------------------------------------+
|1      |Toy Story (1995)                  |Adventure Animation Children Comedy Fantasy pixar fun                  |
|2      |Jumanji (1995)                    |Adventure Children Fantasy robin williams game fantasy magic board game|
|3      |Grumpier Old Men (1995)           |Comedy Romance old moldy                                               |
|4      |Waiting to Exhale (1995)          |Comedy Drama Romance                                                   |
|5      |Father of the Bride Part II (1995)|Comedy remake pregnancy                                                |
+-------+----------------------------------+--------------------

## TF-IDF et similarité cosinus

On transforme le champ `content` de chaque film en vecteur numérique via TF-IDF,
puis on calcule la similarité cosinus entre films pour identifier les plus proches.

In [12]:
# Step 1 : tokenize content string into list of words
tokenizer = Tokenizer(inputCol="content", outputCol="words")

# Step 2 : compute term frequency (TF)
# numFeatures=2048 : hash space size — trade-off between precision and memory
hashing_tf = HashingTF(inputCol="words", outputCol="raw_features", numFeatures=2048)

# Step 3 : compute inverse document frequency (IDF)
idf = IDF(inputCol="raw_features", outputCol="features", minDocFreq=2)

# Build and fit pipeline
pipeline = Pipeline(stages=[tokenizer, hashing_tf, idf])
tfidf_model = pipeline.fit(movies_with_content)
movies_tfidf = tfidf_model.transform(movies_with_content)

movies_tfidf.select("movieId", "title", "features").show(3, truncate=True)
print(f"TF-IDF vectors computed for {movies_tfidf.count():,} movies ✓")

+-------+--------------------+--------------------+
|movieId|               title|            features|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|(2048,[185,241,55...|
|      2|      Jumanji (1995)|(2048,[185,241,64...|
|      3|Grumpier Old Men ...|(2048,[997,1608,1...|
+-------+--------------------+--------------------+
only showing top 3 rows
TF-IDF vectors computed for 9,708 movies ✓


In [15]:
# Collect vectors to driver — acceptable for 9k movies
# For 32M dataset this would need a different approach
movies_vectors = movies_tfidf.select("movieId", "title", "features").collect()

# Build lookup dict : movieId → (title, vector)
movie_index = {
    row.movieId: (row.title, row.features)
    for row in movies_vectors
}

def cosine_similarity(v1, v2):
    """Compute cosine similarity between two sparse vectors."""
    v1_dense = np.array(v1.toArray())
    v2_dense = np.array(v2.toArray())
    norm1 = np.linalg.norm(v1_dense)
    norm2 = np.linalg.norm(v2_dense)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return float(np.dot(v1_dense, v2_dense) / (norm1 * norm2))

def get_similar_movies(movie_id, top_n=10):
    """Return top_n most similar movies to a given movieId."""
    if movie_id not in movie_index:
        print(f"movieId {movie_id} not found")
        return []

    target_title, target_vector = movie_index[movie_id]
    scores = []

    for mid, (title, vector) in movie_index.items():
        if mid == movie_id:
            continue
        sim = cosine_similarity(target_vector, vector)
        scores.append((mid, title, sim))

    scores.sort(key=lambda x: x[2], reverse=True)
    return scores[:top_n]

# Test : films similaires à Toy Story (movieId=1)
print("=== Films similaires à Toy Story (1995) ===")
for mid, title, score in get_similar_movies(1, top_n=10):
    print(f"  {score:.4f}  {title}")

=== Films similaires à Toy Story (1995) ===
  0.7597  Bug's Life, A (1998)
  0.6074  Guardians of the Galaxy 2 (2017)
  0.4299  Antz (1998)
  0.4299  Adventures of Rocky and Bullwinkle, The (2000)
  0.4299  Emperor's New Groove, The (2000)
  0.4299  Monsters, Inc. (2001)
  0.4299  Wild, The (2006)
  0.4299  Shrek the Third (2007)
  0.4299  Tale of Despereaux, The (2008)
  0.4299  Asterix and the Vikings (Astérix et les Vikings) (2006)


### Recommandations content-based pour utilisateurs fictifs

Pour chaque utilisateur, on identifie ses films préférés (note ≥ 4.0),
on calcule les films les plus similaires via cosine similarity,
puis on agrège les scores en excluant les films déjà notés.

In [16]:
def recommend_content_based(user_id, top_n=10, min_rating=4.0):
    """
    Recommend movies for a user based on content similarity.
    
    Args:
        user_id   : target user
        top_n     : number of recommendations to return
        min_rating: minimum rating to consider a movie "liked"
    """
    # Get movies liked by this user
    liked = (
        ratings_clean
        .filter((col("userId") == user_id) & (col("rating") >= min_rating))
        .select("movieId", "rating")
        .collect()
    )

    if not liked:
        print(f"No ratings >= {min_rating} found for user {user_id}")
        return

    liked_ids = {row.movieId for row in liked}

    # Aggregate similarity scores across all liked movies
    scores = {}
    for row in liked:
        if row.movieId not in movie_index:
            continue
        similars = get_similar_movies(row.movieId, top_n=50)
        for mid, title, sim in similars:
            if mid in liked_ids:
                continue  # skip already seen movies
            if mid not in scores:
                scores[mid] = {"title": title, "score": 0.0, "count": 0}
            # Weight similarity by the user's rating of the source movie
            scores[mid]["score"] += sim * row.rating
            scores[mid]["count"] += 1

    # Sort by aggregated score
    ranked = sorted(scores.items(), key=lambda x: x[1]["score"], reverse=True)

    print(f"\n=== Content-based recommendations for user {user_id} ===")
    print(f"    (based on {len(liked)} liked movies)\n")
    for mid, data in ranked[:top_n]:
        print(f"  {data['score']:6.3f}  {data['title']}")

# Test on 5 users
for uid in [1, 42, 100, 200, 500]:
    recommend_content_based(uid, top_n=10)


=== Content-based recommendations for user 1 ===
    (based on 200 liked movies)

  53.789  Quest for Camelot (1998)
  52.768  Cats Don't Dance (1997)
  52.768  Many Adventures of Winnie the Pooh, The (1977)
  50.975  Rudolph, the Red-Nosed Reindeer (1964)
  50.160  Land Before Time III: The Time of the Great Giving (1995)
  50.160  Rock-A-Doodle (1991)
  50.088  Peter Pan (1953)
  50.088  Beauty and the Beast: The Enchanted Christmas (1997)
  50.088  Strange Magic (2015)
  49.776  Frosty the Snowman (1969)

=== Content-based recommendations for user 42 ===
    (based on 259 liked movies)

  87.680  Four Rooms (1995)
  87.680  Ace Ventura: When Nature Calls (1995)
  87.680  Bio-Dome (1996)
  87.680  Friday (1995)
  87.680  Black Sheep (1996)
  87.680  Mr. Wrong (1996)
  87.680  Steal Big, Steal Little (1995)
  87.680  Flirting With Disaster (1996)
  87.680  Down Periscope (1996)
  87.680  Birdcage, The (1996)

=== Content-based recommendations for user 100 ===
    (based on 107 liked 

### Résultats et limites du content-based

Le modèle produit des recommandations cohérentes avec les goûts observés :
- User 1 (200 films aimés) → films d'animation/enfants
- User 42 (259 films aimés) → comédies
- User 100 (107 films aimés) → drames romantiques

**Limite observée** : de nombreux films obtiennent des scores identiques car
ils partagent uniquement les genres sans tags distinctifs. Le TF-IDF ne peut
pas départager des films qui ont exactement le même profil de genres.

Cette limite est inhérente à la richesse des tags disponibles : seulement
3 574 tags pour 9 708 films — environ 65% des films n'ont aucun tag.
Plus le catalogue est taggé, meilleure sera la discrimination.

## Recommandation par proximité utilisateurs (KNN)

### Principe

Le KNN (K-Nearest Neighbors) identifie les utilisateurs les plus similaires
à un utilisateur cible en comparant leurs historiques de notes.

Chaque utilisateur est représenté par un vecteur sparse :
- Dimensions = films du catalogue
- Valeurs = notes données (0 si film non noté)

La similarité cosinus mesure l'angle entre deux vecteurs utilisateurs.
Deux users qui ont noté les mêmes films avec les mêmes notes auront
une similarité proche de 1.0.

### Différence clé avec ALS

ALS apprend un modèle global (facteurs latents) sur tout le dataset.
KNN ne fait aucun apprentissage — il calcule des distances à la demande.
C'est plus lent mais plus interprétable : on peut expliquer pourquoi
un film est recommandé ("parce que 5 users similaires à toi l'ont adoré").

### Hyperparamètre clé : K

K = nombre de voisins à considérer.
- K trop petit → recommandations trop personnalisées, sensibles au bruit
- K trop grand → recommandations trop génériques, perd la personnalisation
On testera K ∈ {5, 10, 20}.